In [1]:
import cv2
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [2]:
IMAGE_SIZE = 128
PATCH_SIZE = 16

NUM_CLASSES = 38

BATCH_SIZE = 8
EPOCHS = 10

PROJECTION_DIM = 64
NUM_HEADS = 4
TRANSFORMER_LAYERS = 4

# =========================
# LOAD MODEL
# =========================

In [24]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "./train",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

Found 70295 files belonging to 38 classes.
Using 56236 files for training.


In [25]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    "./valid",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

Found 17572 files belonging to 38 classes.
Using 3514 files for validation.


In [26]:
class_names = train_ds.class_names

# Normalize

In [27]:
from tensorflow.keras import layers

In [7]:
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

val_ds = val_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

# =========================
# PATCH LAYER
# =========================

In [28]:
class Patches(layers.Layer):

    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size,
                   self.patch_size, 1],

            strides=[1, self.patch_size,
                     self.patch_size, 1],

            rates=[1,1,1,1],
            padding="VALID"
        )

        patch_dims = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [batch_size, -1, patch_dims]
        )

        return patches

    def get_config(self):
        config = super().get_config()
        config.update({
            "patch_size": self.patch_size,
        })
        return config

# =========================
# PATCH ENCODER
# =========================

In [29]:
class PatchEncoder(layers.Layer):

    def __init__(self, num_patches,
                 projection_dim):

        super().__init__()

        self.num_patches = num_patches
        self.projection_dim = projection_dim

        self.projection = layers.Dense(
            projection_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patch):

        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1
        )

        encoded = (
            self.projection(patch)
            + self.position_embedding(positions)
        )

        return encoded

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim,
        })
        return config

## =========================
### BUILD MODEL
## =========================

In [30]:
num_patches = (
    IMAGE_SIZE // PATCH_SIZE
) ** 2

inputs = layers.Input(
    shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

patches = Patches(PATCH_SIZE)(inputs)

encoded_patches = PatchEncoder(
    num_patches,
    PROJECTION_DIM
)(patches)

### Transformer blocks

In [31]:
for _ in range(TRANSFORMER_LAYERS):

    x1 = layers.LayerNormalization(
        epsilon=1e-6
    )(encoded_patches)

    attention_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=PROJECTION_DIM,
        dropout=0.1
    )(x1, x1)

    x2 = layers.Add()(
        [attention_output, encoded_patches]
    )

    x3 = layers.LayerNormalization(
        epsilon=1e-6
    )(x2)

    x3 = layers.Dense(
        PROJECTION_DIM * 2,
        activation=tf.nn.gelu
    )(x3)

    x3 = layers.Dropout(0.1)(x3)

    x3 = layers.Dense(PROJECTION_DIM)(x3)

    encoded_patches = layers.Add()([x3, x2])

## Classification Head

In [32]:
representation = layers.LayerNormalization(
    epsilon=1e-6
)(encoded_patches)

representation = layers.Flatten()(representation)

representation = layers.Dropout(0.3)(
    representation
)

features = layers.Dense(
    128,
    activation=tf.nn.gelu
)(representation)

features = layers.Dropout(0.3)(features)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(features)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)


## Compile Model

In [33]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

## Train Model

In [14]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/10
7030/7030 [==============================] - 627s 88ms/step - loss: 3.6399 - accuracy: 0.0281 - val_loss: 3.6363 - val_accuracy: 0.0299
Epoch 2/10
7030/7030 [==============================] - 506s 72ms/step - loss: 3.6378 - accuracy: 0.0278 - val_loss: 3.6363 - val_accuracy: 0.0299
Epoch 3/10
7030/7030 [==============================] - 495s 70ms/step - loss: 3.6367 - accuracy: 0.0281 - val_loss: 3.6364 - val_accuracy: 0.0299
Epoch 4/10
7030/7030 [==============================] - 515s 73ms/step - loss: 3.6367 - accuracy: 0.0279 - val_loss: 3.6364 - val_accuracy: 0.0299
Epoch 5/10
7030/7030 [==============================] - 539s 77ms/step - loss: 3.6367 - accuracy: 0.0275 - val_loss: 3.6364 - val_accuracy: 0.0299
Epoch 6/10
7030/7030 [==============================] - 535s 76ms/step - loss: 3.6367 - accuracy: 0.0276 - val_loss: 3.6364 - val_accuracy: 0.0299
Epoch 7/10
7030/7030 [==============================] - 535s 76ms/step - loss: 3.6367 - accuracy: 0.0280 - val_loss: 3

## Save Model

In [34]:
model.save(
    "vit_plant_disease_model.h5"
)

print("MODEL SAVED!")

MODEL SAVED!


## Accuracy Graph

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend([
    "Train",
    "Validation"
])

plt.show()